# Translate& Summarize News Article Using Open Source Models - Ollama locally 

In this notebook i experimented ollama by downloading and running it in my own PC and checked on how it translates and summarizes the scrapped chinese website text while utilizing a small model of it.

##### First, I checked whether Ollama is runnig locally in my PC

In [3]:
requests.get("http://localhost:11434").content

b'Ollama is running'

##### Then I downloaded the **Llama 3.2** model, which is available in 1B and 3B parameter variants. It is optimized for multilingual dialogue, agentic retrieval, and summarization tasks. More information is available in the [Ollama Llama 3.2 documentation](https://ollama.com/library/llama3.2).

In [1]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 121 KB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 1.0 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 1.3 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 1.6 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.8 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 4.6 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 5.0 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕         

In [ ]:
# Reusing the article text scraped in Day 1.

from pathlib import Path
import importlib
import sys
import urllib.request
import json

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import src.services.scraper as scraper_module
scraper_module = importlib.reload(scraper_module)
from src.services.scraper import scrape_article

url = "https://www.bbc.com/zhongwen/articles/cjwgxdz5n49o/trad"

if "article" in locals() and article:
    article_text = article
else:
    article_text = scrape_article(url)

print(article_text[:2000])

中國能複製電動車成功經驗到「機器人的士」嗎?
- Author, 蘇蘭嘉娜·特瓦里（Suranjana Tewari）
- Role, BBC 亞洲商務記者
- Reporting from, 北京報導
 
- Published
- 閱讀時間: 4 分鐘
在中國北京亦莊區，無人駕駛車輛已成為常見景象。機器人的士（robotaxi；機器人計程車／機器人出租車）穿梭在車流中，與普通汽車並行，而自動駕駛送貨車則在內側車道平穩行駛，將包裹運送到取件點。
這個區域已成為中國自動駕駛技術的重要測試場之一，百度、文遠知行（WeRide）和小馬智行（Pony.ai）等公司在指定區域內提供商業化的機器人的士服務。
叫車只需打開手機應用程式，短短幾分鐘內，一輛無人的士就會抵達，駕駛座上空無一人。在觸控螢幕上確認行程後，車輛便融入北京繁忙的車流中，穿梭於公車、腳踏車、機車和行人之間，幾乎毫不遲疑。
這項技術仍在持續演進。但，更大的問題已浮現：中國企業能否像主導電動車（EV）市場一樣，讓機器人的士成為另一個全球領先的領域？
自動駕駛車崛起中國市場
中國的自動駕駛企業已擁有強大的優勢，那就是幫助中國成為全球最大電動車市場的產業生態系統。
不過，與大部分技術都由自家設計的特斯拉不同，中國的自駕產業是建立在企業網絡基礎上的。
譬如，比亞迪、奇瑞、吉利和上汽等老牌車廠負責製造車輛，而專業公司則專注開發軟體。
事實上，自動駕駛車輛仰賴與電動車相同的電池、感測器、晶片和車載電腦等關鍵部件。而且，由於這些供應鏈已存在龐大規模，企業得以更快、更低成本地開發技術。
「你看到的是中國電動車產業的創新與適應速度，我認為世界其他地方都難以匹敵。」布魯金斯學會（Brookings Institution）外交政策研究員陳凱欣（Kyle Chan）告訴BBC。
陳凱欣強調，「中國的電動車產能並非僅止於此，它還透過我稱之的『重疊技術產業生態系統』（overlapping tech industrial ecosystems），溢出到其他相關產業。」
新生產力
中國政府政策也扮演重要角色。多個城市的試點計畫允許企業在部分公共道路上測試這項技術。
但是，中國還為致力讓技術更聰明的企業提供了另一項優勢：複雜的駕駛環境。
在北京開一段路，自動駕駛車輛可能就需要應對公車、機車、腳踏車、行人以及難以預測的車流。
「中國這邊的交通

In [ ]:
# Translation with Ollama

def ollama_generate(prompt: str, model: str = "llama3.2", system: str | None = None) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
    }
    if system:
        payload["system"] = system
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as response:
        result = json.loads(response.read().decode("utf-8"))
    return result.get("response", "")

system_prompt = (
    "You are a professional translator. Translate Chinese text to English only. "
    "Do not explain, do not add commentary, and do not leave any Chinese text in the output."
)

translation_prompt = (
    "Translate the following Chinese article into natural English. "
    "Preserve the meaning and key facts. Return only the translated English text.\n\n"
    f"{article_text[:12000]}"
)

translation = ollama_generate(translation_prompt, system=system_prompt)
print(translation[:4000])

Can China's experience with electric cars lead to the development of a robotaxi service?

As Beijing's Yangtze River District becomes a common sight for unmanned taxis, robots are weaving in and out of traffic alongside ordinary cars. Self-driving delivery trucks are also cruising down inner roads, making their way to pick-up points.

This area has become one of China's key test grounds for autonomous driving technology. Baidu, WeRide, and Pony.ai have set up commercial robotaxi services in designated areas.

To hail a ride, all you need to do is open your phone app, and within minutes, an unmanned taxi will arrive at your destination. Once on board, the driverless car seamlessly merges with traffic, navigating alongside buses, bicycles, motorcycles, and pedestrians without hesitation.

But this technology is still evolving. A bigger question is whether China can replicate its success in the electric vehicle market to become a leader in robotaxi services as well?

China's autonomous dr

In [5]:
# Summarize the translated article with Ollama
summary_prompt = (
    "Summarize the following article in 3 bullet points. Keep it concise and clear.\n\n"
    f"{translation if 'translation' in locals() and translation else ''}"
)

if 'translation' not in locals() or not translation:
    translation_prompt = (
        "Translate the following article into natural English. Keep the meaning and key facts intact.\n\n"
        f"{article_text[:12000]}"
    )
    translation = ollama_generate(translation_prompt)

summary = ollama_generate(summary_prompt)
print(summary)

Here are 3 bullet points summarizing the article:

• China's autonomous driving technology has made significant progress, including self-driving cars and buses, with companies like Baidu, WeRide, and Pony.ai offering commercial services in cities like Beijing.

• Despite the advancements, challenges remain, such as security and public trust issues, which can impact the success of Chinese automakers in becoming a global leader in autonomous driving.

• The Chinese government's support for autonomous driving technology through funding and policy incentives provides a boost to companies, but they still need to address regulatory, operational, and public perception hurdles to succeed globally.


##### Here I experimented with the ***DeepSeek-R1:1.5B*** open-source reasoning model. This version is a distilled variant built on Alibaba Cloud's **Qwen** architecture, offering strong reasoning capabilities while remaining lightweight. More information is available on the [Ollama DeepSeek-R1 model page](https://ollama.com/library/deepseek-r1).

In [7]:
!ollama pull deepseek-r1:1.5b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 215 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 950 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 1.2 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 1.7 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 3.3 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 4.3 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 5.1 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 7.2 MB/1.1 GB              

In [8]:
# Translation with Ollama-DeepSeek

def ollama_generate(prompt: str, model: str = "deepseek-r1:1.5b", system: str | None = None) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
    }
    if system:
        payload["system"] = system
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as response:
        result = json.loads(response.read().decode("utf-8"))
    return result.get("response", "")

system_prompt = (
    "You are a professional translator. Translate Chinese text to English only. "
    "Do not explain, do not add commentary, and do not leave any Chinese text in the output."
)

translation_prompt = (
    "Translate the following Chinese article into natural English. "
    "Preserve the meaning and key facts. Return only the translated English text.\n\n"
    f"{article_text[:12000]}"
)

translation = ollama_generate(translation_prompt, system=system_prompt)
print(translation[:4000])

Can China Replicate electric vehicle success stories like "Machinist"?

- Author, Suranjana Tewari
- Role, BBC Asian Business Reporter
- Reporting from Beijing

In Beijing, without driving cars, the millions of machines have become a common sight. Machine people (robotaxi; machine routing vehicles or machine出租ers) pass by in traffic, while automatic drivers (autonomous trucks) drive smoothly through lanes, delivering packages to take-away points.

This region has become an important testing area for autonomous driving technology, with companies like百度, WeRide and Pony.ai providing specialized machine people services in the Beijing area.

The question remains: Can China achieve what EV markets do — making machine people the new specialty?

Machine people have grown in China's market. That is because they are leveraging large strengths, such as having strong leadership roles, to assist China in becoming the largest electric vehicle (EV) industry ecosystem.

But unlike most technology, wh

In [9]:
# Summarize the translated article with Ollama-DeepSeek
summary_prompt = (
    "Summarize the following article in 3 bullet points. Keep it concise and clear.\n\n"
    f"{translation if 'translation' in locals() and translation else ''}"
)

if 'translation' not in locals() or not translation:
    translation_prompt = (
        "Translate the following article into natural English. Keep the meaning and key facts intact.\n\n"
        f"{article_text[:12000]}"
    )
    translation = ollama_generate(translation_prompt)

summary = ollama_generate(summary_prompt)
print(summary)

**Summary of China's Electric Vehicle Success:**

1. **Leverage Technology and Markets:** Automakers in China, like ratio, powertrain, and GM, have used specialized companies to develop software for their machine people, enabling them to test EVs effectively in Beijing.

2. **Challenges and Growth:** Despite relying on technology, Chinese automakers face issues like battery inefficiencies in extreme temperatures and environmental extremes, limiting the scalability of EV development.

3. **Role in International Expansion:** Electric power becomes less critical due to the valuable machine people data, which helps expand software across different markets. China's growing presence is a strategic move toward global EV dominance, influenced by U.S. automakers like Waymo and Tesla.


### Model Comparison: Chinese → English Translation

| Model | Response Time | Translation Quality | Key Observations |
|:------|--------------:|:-------------------:|------------------|
| **Grok Llama 3.3 70B** | **5.4 s** | ⭐⭐⭐⭐⭐ | Most accurate and complete translation. Preserved names, technical terms, quotations, and context with no noticeable hallucinations. |
| **Llama 3.2** | **3 min 39.4 s** | ⭐⭐⭐☆☆ | Produced a reasonable summary but omitted significant portions of the article and contained several translation errors (e.g., incorrect place names). |
| **DeepSeek-R1:1.5B** | **1 min 28.4 s** | ⭐⭐☆☆☆ | Faster than Llama 3.2 but produced numerous mistranslations and hallucinations (e.g., "machine people"), making it unreliable for translation tasks. |


### Key Takeaways

- **Grok Llama 3.3 70B** delivered the **highest translation quality** while also being the **fastest** model.
- **DeepSeek-R1:1.5B** responded faster than Llama 3.2 but sacrificed translation accuracy due to hallucinations and literal translations.
- **Llama 3.2** was the **slowest** model and tended to summarize rather than faithfully translate the source text.
- 📌 **Overall ranking (quality):** Grok Llama 3.3 70B > Llama 3.2 > DeepSeek-R1:1.5B.
- 📌 **Overall ranking (latency):** Grok Llama 3.3 70B > DeepSeek-R1:1.5B > Llama 3.2.